# Actividad 3

Equipo: Panditas Panaderos

In [ ]:
%pip install scikit-learn pandas

# Paso 1: Elección del dataset

Se seleccionará el dataset de Breast Cancer recuperado del siguiente enlace: *https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html*

# Paso 2: Información del dataset

* **Nombre:** *Breast Cancer*
* **Fuente:** *scikit-learn developers. (s.f.). load_breast_cancer.*
* **Liga de acceso:** *https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html*

**Contexto**
Para esta actividad se aplicará un modelo de máquina de soporte vectorial y una técnica de agrupamiento sobre un mismo conjunto de datos con el propósito de:
- comparar las preguntas que responde cada enfoque
- evaluar sus resultados
- interpretar los patrones identificados mediante visualizaciones y métricas pertinentes

*Cada registro representa una muestra de tejido mamario obtenida mendiante una aspiración con aguja fina de una masa sospechosa en un paciente.*

*La variable objetivo es la clasificación del tumor. Indica si la masa es benigna o maligna basándose en el análisis de las células.*

*Las variables predictoras contienen características numéricas calculadas a partir de una imagen digitalizada de la muestra de tejido. Incluyen medidas de los núcleos celulares presentes en la imagen, como el radio, la textura, el perímetro, el área, la suavidad, la compacidad, la concavidad, entre otros.*

**Pregunta para el análisis supervisado**
*¿Se puede predecir con alta precisión si un tumor es benigno o maligno basándose únicamente en las características morfométricas de sus núcleos celulares?*

**Pregunta para el análisis no supervisado**
*¿Existen agrupaciones naturales en los datos que separen las muestras con características de tumores malignos de los benignos sin necesidad de conocer su etiqueta real previamente?*

# Paso 3: Revisión inicial de la estructura

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)

A diferencia de las actividades anteriores donde teníamos la variable objetivo dentro del dataframe, ahora el **dataframe** sólo incluye las **variables predictoras**, mientras que el **data** sí incluye toda la información. Para esta actividad estaremos trabajando con los 2 formatos para realizar una exploración más precisa.

In [ ]:
print("\nEstructura del Dataset Breast Cancer:\n")

# Número de registros
print("Número de registros:", df.shape[0])

# Número de variables
print("Número de variables:", df.shape[1])

# Variables disponibles
print("Variables disponibles:", df.columns.tolist())

# Tipos de datos de cada variable
print("\nTipos de datos de cada variable:\n", df.dtypes)

In [ ]:
# Presencia de valores faltantes
print("\nPresencia de valores faltantes:\n", df.isnull().sum())

In [ ]:
# Posibles duplicados
print("\nNúmero de registros duplicados:", df.duplicated().sum())

En los resultados podemos ver que el dataset está compuesto por 569 registros y 30 variables numéricas predictoras, todas de tipo float64. Asimismo, confirmamos que no existen valores nulos ni registros duplicados. Por esta razón, no es necesario aplicar técnicas de imputación, eliminación de filas ni conversiones de tipo de datos.

In [ ]:
# Distribución de la variable objetivo
print("\nDistribución de la variable objetivo:\n")
print(pd.Series(data.target).value_counts())

print("\nDistribución de la variable objetivo (nombres):\n")
target_names = pd.Series(data.target).map({
  0: data.target_names[0],
  1: data.target_names[1]
})
print(target_names.value_counts())

print("\nDistribución de la variable objetivo (porcentaje):\n")
print(target_names.value_counts(normalize=True) * 100)

Para que quizá sea un poco más fácil la exploración de la variable objetivo, ésta la podemos agregar a una copia del dataset.

In [ ]:
df_target = df.copy()
df_target['target'] = data.target

df_target['target_name'] = pd.Categorical.from_codes(data.target, data.target_names)

print(df_target.head(25))

print("\nDistribución de la variable objetivo (nombres) con el DataFrame:\n")
print(df_target['target_name'].value_counts())

# Paso 4: Selección de variables numericas a utilizar

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Selección de predictoras (X) y separación de la variable objetivo (y)
X = df.copy() 
y = data.target

# Aplicar escalamiento
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

print("Datos escalados exitosamente.")
display(X_scaled.head())

¿Por qué SVM y K-Means se ven afectados por las diferencias de escala?

Básicamente, los dos modelos hacen su trabajo midiendo distancias entre los datos. Por eso, si los números están en escalas muy diferentes, los cálculos se descuadran.

Lo que pasa en K-Means (No Supervisado): Este modelo arma los grupos viendo qué tan cerca están los datos unos de otros. El problema es que si tenemos una variable como el "área" que maneja números de miles, y otra como la "suavidad" que son puros decimales, el algoritmo se va a guiar casi por completo por los números grandes y va a ignorar los pequeños. Escalar los datos sirve para emparejar y que todas las variables pesen lo mismo.

Lo que pasa en SVM (Supervisado): Este algoritmo trata de trazar la mejor línea posible para separar los tipos. Como también se basa en distancias, si no escalamos los datos, la línea se va a deformar dándole preferencia a las variables más grandes. Al usar la herramienta de escalamiento (StandardScaler), ponemos todos los números en una misma proporción; esto evita que el modelo se confunda y le ayuda a encontrar esa separación de forma más rápida y precisa.

# Paso 5: ANÁLISIS SUPERVISADO

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Total de registros para entrenar: {X_train.shape[0]}")
print(f"Total de registros para probar: {X_test.shape[0]}")

¿Por qué dividir los datos nos permite evaluar el modelo con datos no vistos?

Básicamente, hacemos esta división para que el modelo no se aprenda los resultados de memoria.

Lo que pasa con el grupo de entrenamiento (80%): Le damos la mayor parte de los datos al modelo para que los revise, vea las respuestas reales y aprenda a diferenciar los tipos de tumores.

Lo que pasa con el grupo de prueba (20%): Estos son los datos "no vistos". Los separamos desde el principio y se los ocultamos al modelo. El problema es que, si lo evaluamos con los mismos datos que ya estudió, el modelo va a acertar todo. Al ponerlo a prueba con este 20% que nunca ha visto, comprobamos si de verdad aprendió a identificar los patrones. Así estamos seguros de que funcionará bien cuando analice muestras de pacientes totalmente nuevos.

# Paso 6: Entreamiento de modelos y resultados

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Preparamos el primer modelo con kernel lineal
svm_lineal = SVC(kernel='linear', random_state=42)

# Preparamos el segundo modelo con kernel RBF
svm_rbf = SVC(kernel='rbf', random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Volvemos a dividir los datos originales sin escalar (X)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)


pipeline_lineal = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', svm_lineal)
])

pipeline_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', svm_rbf)
])


pipeline_lineal.fit(X_train_raw, y_train)
pipeline_rbf.fit(X_train_raw, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Hacemos las predicciones con los datos de prueba
y_pred_lineal = pipeline_lineal.predict(X_test_raw)
y_pred_rbf = pipeline_rbf.predict(X_test_raw)

# Función para calcular todas las calificaciones
def calcular_metricas(y_real, y_pred):
    return [
        round(accuracy_score(y_real, y_pred), 4),
        round(precision_score(y_real, y_pred), 4),
        round(recall_score(y_real, y_pred), 4),
        round(f1_score(y_real, y_pred), 4)
    ]

resultados_lineal = calcular_metricas(y_test, y_pred_lineal)
resultados_rbf = calcular_metricas(y_test, y_pred_rbf)

In [ ]:
import pandas as pd

# Tabla de resultados
tabla_resultados = pd.DataFrame({
    'Modelo SVM': ['Modelo 1', 'Modelo 2'],
    'Kernel': ['Lineal', 'RBF o polinomial'],
    'Accuracy': [resultados_lineal[0], resultados_rbf[0]],
    'Precision': [resultados_lineal[1], resultados_rbf[1]],
    'Recall': [resultados_lineal[2], resultados_rbf[2]],
    'F1-score': [resultados_lineal[3], resultados_rbf[3]]
})

display(tabla_resultados)

# Paso 7:

# Paso 8:

# Paso 9:

# Paso 10:

# Paso 11:

# Paso 12:

# Paso 13: